# 03 - Neural Network
**Owner:** Syed Ayaan Hasan

A feed-forward neural network (multi-layer perceptron) maps the building features through one or more hidden layers of non-linear units to three output scores, one per damage grade. A softmax turns these scores into class probabilities and the network is trained by backpropagation to minimise cross-entropy loss, with an L2 penalty on the weights (`alpha`) to limit overfitting.

**Design decisions**
- Inputs are the same standardised and one-hot encoded features used for KNN (`build_scaled_preprocessor`, with `age` capped at 100 by `clip_age`). Neural networks train poorly on unscaled inputs.
- Model: scikit-learn `MLPClassifier` with ReLU activations and the Adam optimiser, using mini-batches of 512 buildings.
- Early stopping: 10% of the training split is held out internally and training stops when its accuracy has not improved for 8 epochs. The validation split is therefore not used to decide when to stop.
- Tuned: the network architecture (section 2) and the L2 strength `alpha` (section 3), both selected by micro-F1 on the validation split.
- Random initialisation changes the result from run to run, so the final configuration is trained with three seeds and the mean and standard deviation are reported rather than the best run.

For context, the earlier CS229 project (2021) used a single hidden layer of 10 sigmoid units and reported a micro-F1 of 0.650 on the competition test set. That architecture is included below as a reference point.

In [ ]:
import os, sys
from pathlib import Path

# --- Colab only: clone the repo and move into it ---
REPO = "ML-Project-Earthquake-Damage"
GITHUB_URL = "https://github.com/ayaanGH06/ML-Project-Earthquake-Damage.git"  # private repo: use a token URL in Colab, never commit it

if "google.colab" in sys.modules and not os.path.exists("src"):
    if not os.path.exists(REPO):
        !git clone {GITHUB_URL}
    %cd {REPO}
    !pip install -q -r requirements.txt

# --- Local Jupyter: notebooks run from notebooks/, so go up to the repo root ---
if Path.cwd().name == "notebooks":
    os.chdir("..")

# --- Data: the 3 CSVs must be in data/raw/ (see README for the download link) ---
DATA_DIR = Path("data/raw")
needed = ["train_values.csv", "train_labels.csv", "test_values.csv"]
missing = [f for f in needed if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {DATA_DIR.resolve()}. "
        "Download them from the Google Drive link in the README and place them in data/raw/."
    )

from src.data import load_raw, train_val_split, build_scaled_preprocessor, build_tree_preprocessor
from src.evaluate import evaluate_model, micro_f1

X, y, X_test = load_raw(DATA_DIR)
X_train, X_val, y_train, y_val = train_val_split(X, y)
print(X_train.shape, X_val.shape)

## 1. Setup and feature matrix

In [ ]:
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from src.data import clip_age

FIG_DIR = Path("results/figures")
METRIC_DIR = Path("results/metrics")
for d in (FIG_DIR, METRIC_DIR):
    d.mkdir(parents=True, exist_ok=True)

majority_baseline = y_val.value_counts(normalize=True).max()
print(f"Majority-class baseline: micro-F1 = {majority_baseline:.4f}")

# Preprocessor is fitted on the training split only.
prep = make_pipeline(FunctionTransformer(clip_age), build_scaled_preprocessor(X_train))
Xs_train = prep.fit_transform(X_train)
Xs_val = prep.transform(X_val)
print("Feature matrix:", Xs_train.shape, Xs_val.shape)

In [ ]:
def make_mlp(hidden, alpha=1e-4, seed=42):
    return MLPClassifier(
        hidden_layer_sizes=hidden, activation="relu", solver="adam",
        alpha=alpha, batch_size=512, learning_rate_init=1e-3, max_iter=100,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=8,
        random_state=seed,
    )

def fit_and_score(hidden, alpha=1e-4, seed=42):
    """Train one network and score it on the validation split."""
    mlp = make_mlp(hidden, alpha, seed)
    t0 = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        mlp.fit(Xs_train, y_train)
    pred = mlp.predict(Xs_val)
    return {"model": mlp, "pred": pred, "hidden": hidden, "alpha": alpha, "seed": seed,
            "val_micro_f1": micro_f1(y_val, pred), "epochs": mlp.n_iter_,
            "seconds": time.time() - t0}

## 2. Network architecture
Four architectures are compared with the default regularisation (`alpha = 1e-4`) and the same random seed. The first is the single 10-unit hidden layer from the CS229 project.

In [ ]:
architectures = {
    "10 (CS229 report)": (10,),
    "64": (64,),
    "128-64": (128, 64),
    "256-128-64": (256, 128, 64),
}

arch_runs = {}
for name, hidden in architectures.items():
    arch_runs[name] = fit_and_score(hidden)
    r = arch_runs[name]
    print(f"{name:>18}: val micro-F1 = {r['val_micro_f1']:.4f} | {r['epochs']:3d} epochs | {r['seconds']:.0f} s")

arch_table = pd.DataFrame(
    [{"architecture": n, "val_micro_f1": r["val_micro_f1"], "epochs": r["epochs"],
      "train_seconds": round(r["seconds"])} for n, r in arch_runs.items()]
).set_index("architecture")
print()
print(arch_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, r in arch_runs.items():
    axes[0].plot(r["model"].loss_curve_, label=name)
    axes[1].plot(r["model"].validation_scores_, label=name)
axes[0].set_title("Training loss per epoch")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[1].set_title("Held-out accuracy (early-stopping set) per epoch")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "03_nn_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

Validation micro-F1 rises with network size, from 0.686 for the single 10-unit layer to 0.695 for 64 units, and then flattens: the 128-64 and 256-128-64 networks reach 0.696 and 0.697. The extra capacity buys almost nothing while training time grows from 27 s to 84 s. The curves show why. The larger networks keep reducing their training loss, but the held-out accuracy of the largest network peaks at about epoch 13 and then declines, which is the onset of overfitting; early stopping ends training and scikit-learn restores the weights from the best epoch. The 10-unit network levels off at a visibly lower held-out accuracy, which indicates that it underfits. The differences between the three largest networks (at most 0.002) are comparable to the sampling error of a validation set of this size (about 0.002), so the choice among them is not decisive.

## 3. Regularisation strength
The best architecture from section 2 is retrained with stronger L2 penalties.

In [ ]:
best_arch = max(arch_runs, key=lambda n: arch_runs[n]["val_micro_f1"])
best_hidden = architectures[best_arch]
print(f"Best architecture: {best_arch}  {best_hidden}")

alpha_runs = {1e-4: arch_runs[best_arch]}
for alpha in (1e-3, 1e-2):
    alpha_runs[alpha] = fit_and_score(best_hidden, alpha=alpha)

alpha_table = pd.DataFrame(
    [{"alpha": a, "val_micro_f1": r["val_micro_f1"], "epochs": r["epochs"]}
     for a, r in sorted(alpha_runs.items())]
).set_index("alpha")
print(alpha_table.round(4))

best_alpha = max(alpha_runs, key=lambda a: alpha_runs[a]["val_micro_f1"])
best_run = alpha_runs[best_alpha]
print(f"\nSelected: architecture {best_arch}, alpha = {best_alpha}")

Changing the L2 penalty from 1e-4 to 1e-2 moves the validation micro-F1 by less than 0.001 (between 0.6964 and 0.6972), so the regularisation strength has no meaningful effect here; early stopping already limits overfitting. alpha = 0.01 is selected as the nominal best.

## 4. Final model: three random seeds
Because the weights are initialised randomly, the selected configuration is trained with two further seeds. The reported score is the mean over the three runs; the confusion matrix and the saved metrics file refer to the first run (seed 42).

In [ ]:
final_runs = [best_run] + [fit_and_score(best_hidden, best_alpha, seed=s) for s in (1, 2)]
scores = np.array([r["val_micro_f1"] for r in final_runs])
for r in final_runs:
    print(f"seed {r['seed']:>2}: val micro-F1 = {r['val_micro_f1']:.4f} ({r['epochs']} epochs)")
print(f"\nmean = {scores.mean():.4f}, std = {scores.std():.4f}, min = {scores.min():.4f}, max = {scores.max():.4f}")

In [ ]:
nn_f1 = evaluate_model("neural_network", y_val, best_run["pred"])

In [ ]:
with open(METRIC_DIR / "neural_network_params.json", "w") as f:
    json.dump({"hidden_layer_sizes": list(best_hidden), "alpha": best_alpha,
               "val_micro_f1_seed42": float(nn_f1),
               "val_micro_f1_mean_3_seeds": float(scores.mean()),
               "val_micro_f1_std_3_seeds": float(scores.std()),
               "majority_baseline": float(majority_baseline)}, f, indent=2)

print(f"Neural network {best_hidden}, alpha={best_alpha}")
print(f"Validation micro-F1 (mean of 3 seeds): {scores.mean():.4f} +/- {scores.std():.4f}")
print(f"Majority-class baseline:               {majority_baseline:.4f}")
print(f"Improvement over baseline:             {scores.mean() - majority_baseline:+.4f}")

Across three seeds the selected network (256-128-64, alpha = 0.01) reaches a validation micro-F1 of 0.6971 with a standard deviation of 0.0001, so the result is stable with respect to the random initialisation. This is 0.128 above the majority-class baseline (0.5689) and 0.017 above KNN (0.6802), but still below the untuned Random Forest baseline in `main.py` (0.7239). The same architecture as the CS229 project (a single hidden layer of 10 units) scores 0.686 here, compared with 0.650 reported there; the two numbers are not directly comparable (validation split versus competition test set, and different preprocessing and optimiser).

The per-class results follow the same pattern as KNN but are somewhat better balanced: recall is 0.44 for grade 1, 0.81 for grade 2 and 0.57 for grade 3 (KNN: 0.40, 0.82 and 0.52). Most errors are again predictions of the majority class: 2,752 of the 5,025 grade-1 buildings (55%) and 7,352 of the 17,444 grade-3 buildings (42%) are predicted as grade 2. Confusing the two extremes is rare (76 grade-1 buildings predicted as grade 3 and 114 grade-3 buildings predicted as grade 1).

## 5. Notes on the method
- The architecture and `alpha` were chosen using the validation split, so the reported score is slightly optimistic relative to completely unseen data. Differences between configurations of about 0.002 or less are within the sampling error of a validation set of this size and should not be over-interpreted.
- A neural network is not interpretable in the way a decision tree is, and it needs scaled inputs, which is why it uses the one-hot encoded and standardised matrix rather than the compact ordinal encoding used for the tree models.
- Geo levels 2 and 3 are not used (see notebook 01), so the network cannot exploit fine-grained location, which tree models can.